In [ ]:
"""
Bài 3: Thuật toán Minimax cho TicTacToe NxN
Hỗ trợ bàn cờ 5x5, 10x10, NxN với K quân liên tiếp để thắng
"""

import math
import time

# ─────────────────────────────────────────
#  CẤU TRÚC BÀN CỜ
# ─────────────────────────────────────────
def create_board(n):
    return [['.' for _ in range(n)] for _ in range(n)]

def print_board(board):
    n = len(board)
    header = "   " + "  ".join(str(c) for c in range(n))
    print(header)
    print("   " + "─" * (n * 3))
    for r in range(n):
        row = f"{r:2d}│ " + "  ".join(board[r][c] for c in range(n))
        print(row)
    print()

def get_empty_cells(board):
    n = len(board)
    return [(r, c) for r in range(n) for c in range(n) if board[r][c] == '.']

def make_move(board, r, c, player):
    board[r][c] = player

def undo_move(board, r, c):
    board[r][c] = '.'

# ─────────────────────────────────────────
#  KIỂM TRA THẮNG
# ─────────────────────────────────────────
DIRECTIONS = [(0, 1), (1, 0), (1, 1), (1, -1)]

def check_win(board, r, c, player, k):
    """Kiểm tra xem player có k quân liên tiếp qua ô (r,c) không"""
    n = len(board)
    for dr, dc in DIRECTIONS:
        count = 1
        for d in range(1, k):
            nr, nc = r + d*dr, c + d*dc
            if 0 <= nr < n and 0 <= nc < n and board[nr][nc] == player:
                count += 1
            else:
                break
        for d in range(1, k):
            nr, nc = r - d*dr, c - d*dc
            if 0 <= nr < n and 0 <= nc < n and board[nr][nc] == player:
                count += 1
            else:
                break
        if count >= k:
            return True
    return False

def is_board_full(board):
    return all(board[r][c] != '.' for r in range(len(board)) for c in range(len(board[0])))

# ─────────────────────────────────────────
#  HÀM ĐÁNH GIÁ HEURISTIC
# ─────────────────────────────────────────
def evaluate(board, k, ai_player, human_player):
    """Đánh giá điểm trạng thái bàn cờ"""
    n = len(board)
    score = 0

    def score_line(cells):
        ai_count = cells.count(ai_player)
        hu_count = cells.count(human_player)
        empty = cells.count('.')
        if ai_count > 0 and hu_count > 0:
            return 0
        if ai_count > 0:
            return 10 ** ai_count
        if hu_count > 0:
            return -(10 ** hu_count) * 1.1
        return 0

    for r in range(n):
        for c in range(n):
            for dr, dc in DIRECTIONS:
                end_r = r + (k-1)*dr
                end_c = c + (k-1)*dc
                if 0 <= end_r < n and 0 <= end_c < n:
                    cells = [board[r + i*dr][c + i*dc] for i in range(k)]
                    score += score_line(cells)
    return score

# ─────────────────────────────────────────
#  LẤY CÁC NƯỚC ĐI ƯU TIÊN (gần quân đã đi)
# ─────────────────────────────────────────
def get_candidate_moves(board, radius=2):
    n = len(board)
    candidates = set()
    has_piece = False
    for r in range(n):
        for c in range(n):
            if board[r][c] != '.':
                has_piece = True
                for dr in range(-radius, radius+1):
                    for dc in range(-radius, radius+1):
                        nr, nc = r+dr, c+dc
                        if 0 <= nr < n and 0 <= nc < n and board[nr][nc] == '.':
                            candidates.add((nr, nc))
    if not has_piece:
        mid = n // 2
        return [(mid, mid)]
    return list(candidates)

# ─────────────────────────────────────────
#  MINIMAX (THUẦN — BÀI 3)
# ─────────────────────────────────────────
nodes_visited = 0

def minimax(board, depth, is_maximizing, ai_player, human_player, k,
            last_r=-1, last_c=-1):
    """
    Minimax không cắt tỉa.
    Trả về điểm số của trạng thái.
    """
    global nodes_visited
    nodes_visited += 1

    current = ai_player if is_maximizing else human_player
    opponent = human_player if is_maximizing else ai_player

    # Kiểm tra nước đi vừa rồi có thắng không
    if last_r >= 0 and check_win(board, last_r, last_c, opponent, k):
        return (-1000 - depth) if is_maximizing else (1000 + depth)

    if depth == 0 or is_board_full(board):
        return evaluate(board, k, ai_player, human_player)

    moves = get_candidate_moves(board)
    if not moves:
        return 0

    if is_maximizing:
        best = -math.inf
        for r, c in moves:
            make_move(board, r, c, current)
            val = minimax(board, depth-1, False, ai_player, human_player, k, r, c)
            undo_move(board, r, c)
            best = max(best, val)
        return best
    else:
        best = math.inf
        for r, c in moves:
            make_move(board, r, c, current)
            val = minimax(board, depth-1, True, ai_player, human_player, k, r, c)
            undo_move(board, r, c)
            best = min(best, val)
        return best

def find_best_move_minimax(board, depth, ai_player, human_player, k):
    global nodes_visited
    nodes_visited = 0
    best_score = -math.inf
    best_move = None
    t0 = time.time()

    for r, c in get_candidate_moves(board):
        make_move(board, r, c, ai_player)
        score = minimax(board, depth-1, False, ai_player, human_player, k, r, c)
        undo_move(board, r, c)
        if score > best_score:
            best_score = score
            best_move = (r, c)

    elapsed = time.time() - t0
    print(f"  [Minimax] Nút duyệt: {nodes_visited:,} | Thời gian: {elapsed:.3f}s | Điểm: {best_score}")
    return best_move

# ─────────────────────────────────────────
#  GAME LOOP
# ─────────────────────────────────────────
def play_game(n=5, k=4, depth=3, ai_player='O', human_player='X'):
    """Chơi game người vs AI (Minimax)"""
    board = create_board(n)
    print(f"\n{'='*50}")
    print(f"  TicTacToe {n}x{n} — Thắng khi {k} quân liên tiếp")
    print(f"  Thuật toán: MINIMAX (độ sâu={depth})")
    print(f"  Người: {human_player} | AI: {ai_player}")
    print(f"{'='*50}\n")
    print_board(board)

    current = 'X'  # X đi trước
    while True:
        if current == human_player:
            # Lượt người
            while True:
                try:
                    inp = input(f"Lượt {human_player} — Nhập hàng cột (vd: 2 3): ").split()
                    r, c = int(inp[0]), int(inp[1])
                    if 0 <= r < n and 0 <= c < n and board[r][c] == '.':
                        break
                    print("  Ô không hợp lệ, thử lại.")
                except (ValueError, IndexError):
                    print("  Nhập sai định dạng, thử lại.")
            make_move(board, r, c, human_player)
        else:
            # Lượt AI
            print(f"Lượt AI ({ai_player}) đang tính...")
            r, c = find_best_move_minimax(board, depth, ai_player, human_player, k)
            make_move(board, r, c, ai_player)
            print(f"  AI đi: hàng {r}, cột {c}")

        print_board(board)

        if check_win(board, r, c, current, k):
            if current == human_player:
                print("🎉 Bạn thắng!")
            else:
                print("🤖 AI thắng!")
            return

        if is_board_full(board):
            print("🤝 Hoà!")
            return

        current = 'O' if current == 'X' else 'X'

# ─────────────────────────────────────────
#  DEMO CHẠY THỬ AI vs AI
# ─────────────────────────────────────────
def demo_ai_vs_ai(n=5, k=4, depth=2):
    """Demo AI tự chơi với nhau để thấy số nút duyệt"""
    board = create_board(n)
    print(f"\n{'='*50}")
    print(f"  DEMO AI vs AI — Minimax {n}x{n}, thắng {k}, độ sâu {depth}")
    print(f"{'='*50}\n")
    print_board(board)

    players = ['X', 'O']
    idx = 0
    move_count = 0

    while True:
        current = players[idx]
        ai = current
        opp = players[1 - idx]
        r, c = find_best_move_minimax(board, depth, ai, opp, k)
        make_move(board, r, c, current)
        move_count += 1
        print(f"Nước {move_count}: {current} đi ({r},{c})")
        print_board(board)

        if check_win(board, r, c, current, k):
            print(f"Kết quả: {current} thắng!")
            break
        if is_board_full(board):
            print("Kết quả: Hoà!")
            break
        idx = 1 - idx

if __name__ == '__main__':
    import sys
    print("Bài 3 — Minimax TicTacToe NxN")
    print("1. Người vs AI (5x5)")
    print("2. Demo AI vs AI (5x5)")
    print("3. Demo AI vs AI (10x10)")
    choice = input("Chọn (1/2/3): ").strip()
    if choice == '1':
        play_game(n=5, k=4, depth=3)
    elif choice == '2':
        demo_ai_vs_ai(n=5, k=4, depth=2)
    elif choice == '3':
        demo_ai_vs_ai(n=10, k=5, depth=2)
    else:
        demo_ai_vs_ai(n=5, k=4, depth=2)

Bài 3 — Minimax TicTacToe NxN
1. Người vs AI (5x5)
2. Demo AI vs AI (5x5)
3. Demo AI vs AI (10x10)
Chọn (1/2/3): 1

  TicTacToe 5x5 — Thắng khi 4 quân liên tiếp
  Thuật toán: MINIMAX (độ sâu=3)
  Người: X | AI: O

   0  1  2  3  4
   ───────────────
 0│ .  .  .  .  .
 1│ .  .  .  .  .
 2│ .  .  .  .  .
 3│ .  .  .  .  .
 4│ .  .  .  .  .

Lượt X — Nhập hàng cột (vd: 2 3): 2 3
   0  1  2  3  4
   ───────────────
 0│ .  .  .  .  .
 1│ .  .  .  .  .
 2│ .  .  .  X  .
 3│ .  .  .  .  .
 4│ .  .  .  .  .

Lượt AI (O) đang tính...
  [Minimax] Nút duyệt: 8,135 | Thời gian: 0.300s | Điểm: 87.99999999999999
  AI đi: hàng 2, cột 2
   0  1  2  3  4
   ───────────────
 0│ .  .  .  .  .
 1│ .  .  .  .  .
 2│ .  .  O  X  .
 3│ .  .  .  .  .
 4│ .  .  .  .  .

Lượt X — Nhập hàng cột (vd: 2 3): 3 3
   0  1  2  3  4
   ───────────────
 0│ .  .  .  .  .
 1│ .  .  .  .  .
 2│ .  .  O  X  .
 3│ .  .  .  X  .
 4│ .  .  .  .  .

Lượt AI (O) đang tính...
  [Minimax] Nút duyệt: 9,724 | Thời gian: 0.365s | Điể

In [ ]:
"""
Bài 4: Thuật toán Alpha-Beta Pruning cho TicTacToe NxN
So sánh hiệu năng với Minimax thuần (Bài 3)
"""

import math
import time

# ─────────────────────────────────────────
#  BÀN CỜ — dùng lại từ Bài 3
# ─────────────────────────────────────────
def create_board(n):
    return [['.' for _ in range(n)] for _ in range(n)]

def print_board(board):
    n = len(board)
    print("   " + "  ".join(str(c) for c in range(n)))
    print("   " + "─" * (n * 3))
    for r in range(n):
        print(f"{r:2d}│ " + "  ".join(board[r][c] for c in range(n)))
    print()

def get_empty_cells(board):
    n = len(board)
    return [(r, c) for r in range(n) for c in range(n) if board[r][c] == '.']

def make_move(board, r, c, player):
    board[r][c] = player

def undo_move(board, r, c):
    board[r][c] = '.'

def is_board_full(board):
    return all(board[r][c] != '.' for r in range(len(board)) for c in range(len(board[0])))

DIRECTIONS = [(0, 1), (1, 0), (1, 1), (1, -1)]

def check_win(board, r, c, player, k):
    n = len(board)
    for dr, dc in DIRECTIONS:
        count = 1
        for d in range(1, k):
            nr, nc = r + d*dr, c + d*dc
            if 0 <= nr < n and 0 <= nc < n and board[nr][nc] == player:
                count += 1
            else:
                break
        for d in range(1, k):
            nr, nc = r - d*dr, c - d*dc
            if 0 <= nr < n and 0 <= nc < n and board[nr][nc] == player:
                count += 1
            else:
                break
        if count >= k:
            return True
    return False

# ─────────────────────────────────────────
#  HEURISTIC NÂNG CAO
# ─────────────────────────────────────────
def evaluate(board, k, ai_player, human_player):
    n = len(board)
    score = 0

    def score_window(cells):
        ai_c = cells.count(ai_player)
        hu_c = cells.count(human_player)
        em_c = cells.count('.')
        if ai_c > 0 and hu_c > 0:
            return 0
        if ai_c == k:
            return 100000
        if hu_c == k:
            return -100000
        if ai_c > 0:
            return 10 ** ai_c * (1 + em_c * 0.1)
        if hu_c > 0:
            return -(10 ** hu_c) * 1.2 * (1 + em_c * 0.1)
        return 0

    for r in range(n):
        for c in range(n):
            for dr, dc in DIRECTIONS:
                end_r = r + (k-1)*dr
                end_c = c + (k-1)*dc
                if 0 <= end_r < n and 0 <= end_c < n:
                    cells = [board[r + i*dr][c + i*dc] for i in range(k)]
                    score += score_window(cells)
    return score

def get_candidate_moves(board, radius=2):
    """Chỉ xét nước đi gần quân đã có trên bàn (tối ưu không gian tìm kiếm)"""
    n = len(board)
    candidates = set()
    has_piece = False
    for r in range(n):
        for c in range(n):
            if board[r][c] != '.':
                has_piece = True
                for dr in range(-radius, radius+1):
                    for dc in range(-radius, radius+1):
                        nr, nc = r+dr, c+dc
                        if 0 <= nr < n and 0 <= nc < n and board[nr][nc] == '.':
                            candidates.add((nr, nc))
    if not has_piece:
        mid = n // 2
        return [(mid, mid)]
    return list(candidates)

def move_priority(board, r, c, ai_player, human_player, k):
    """Ưu tiên nước đi có giá trị cao hơn để Alpha-Beta cắt tỉa sớm hơn"""
    n = len(board)
    score = 0
    for dr, dc in DIRECTIONS:
        for pl, mult in [(ai_player, 1), (human_player, 1.1)]:
            count = 1
            for d in range(1, k):
                nr, nc = r + d*dr, c + d*dc
                if 0 <= nr < n and 0 <= nc < n and board[nr][nc] == pl:
                    count += 1
                else:
                    break
            for d in range(1, k):
                nr, nc = r - d*dr, c - d*dc
                if 0 <= nr < n and 0 <= nc < n and board[nr][nc] == pl:
                    count += 1
                else:
                    break
            score += (10 ** count) * mult
    return score

# ─────────────────────────────────────────
#  MINIMAX THUẦN (để so sánh)
# ─────────────────────────────────────────
minimax_nodes = 0

def minimax(board, depth, is_max, ai, human, k, last_r=-1, last_c=-1):
    global minimax_nodes
    minimax_nodes += 1
    opp = human if is_max else ai
    if last_r >= 0 and check_win(board, last_r, last_c, opp, k):
        return (-1000 - depth) if is_max else (1000 + depth)
    if depth == 0 or is_board_full(board):
        return evaluate(board, k, ai, human)
    moves = get_candidate_moves(board)
    if not moves:
        return 0
    cur = ai if is_max else human
    if is_max:
        best = -math.inf
        for r, c in moves:
            make_move(board, r, c, cur)
            best = max(best, minimax(board, depth-1, False, ai, human, k, r, c))
            undo_move(board, r, c)
        return best
    else:
        best = math.inf
        for r, c in moves:
            make_move(board, r, c, cur)
            best = min(best, minimax(board, depth-1, True, ai, human, k, r, c))
            undo_move(board, r, c)
        return best

# ─────────────────────────────────────────
#  ALPHA-BETA PRUNING (BÀI 4)
# ─────────────────────────────────────────
alphabeta_nodes = 0

def alphabeta(board, depth, is_max, alpha, beta, ai, human, k,
              last_r=-1, last_c=-1):
    """
    Minimax với cắt tỉa Alpha-Beta.
    alpha: giá trị tốt nhất Max đã tìm được (khởi tạo -inf)
    beta:  giá trị tốt nhất Min đã tìm được (khởi tạo +inf)
    Cắt khi beta <= alpha.
    """
    global alphabeta_nodes
    alphabeta_nodes += 1

    opp = human if is_max else ai
    # Kiểm tra nước vừa đi có thắng không
    if last_r >= 0 and check_win(board, last_r, last_c, opp, k):
        return (-1000 - depth) if is_max else (1000 + depth)

    if depth == 0 or is_board_full(board):
        return evaluate(board, k, ai, human)

    moves = get_candidate_moves(board)
    if not moves:
        return 0

    cur = ai if is_max else human

    # Sắp xếp nước đi để tăng hiệu quả cắt tỉa
    moves_sorted = sorted(
        moves,
        key=lambda m: move_priority(board, m[0], m[1], ai, human, k),
        reverse=is_max
    )

    if is_max:
        best = -math.inf
        for r, c in moves_sorted:
            make_move(board, r, c, cur)
            val = alphabeta(board, depth-1, False, alpha, beta, ai, human, k, r, c)
            undo_move(board, r, c)
            best = max(best, val)
            alpha = max(alpha, best)
            if beta <= alpha:
                break  # CẮT TỈA BETA
        return best
    else:
        best = math.inf
        for r, c in moves_sorted:
            make_move(board, r, c, cur)
            val = alphabeta(board, depth-1, True, alpha, beta, ai, human, k, r, c)
            undo_move(board, r, c)
            best = min(best, val)
            beta = min(beta, best)
            if beta <= alpha:
                break  # CẮT TỈA ALPHA
        return best

# ─────────────────────────────────────────
#  TÌM NƯỚC ĐI TỐT NHẤT
# ─────────────────────────────────────────
def find_best_move(board, depth, ai, human, k, use_alphabeta=True):
    global minimax_nodes, alphabeta_nodes
    minimax_nodes = 0
    alphabeta_nodes = 0

    best_score = -math.inf
    best_move = None
    t0 = time.time()
    moves = get_candidate_moves(board)
    moves_sorted = sorted(
        moves,
        key=lambda m: move_priority(board, m[0], m[1], ai, human, k),
        reverse=True
    )

    for r, c in moves_sorted:
        make_move(board, r, c, ai)
        if use_alphabeta:
            score = alphabeta(board, depth-1, False, -math.inf, math.inf,
                              ai, human, k, r, c)
        else:
            score = minimax(board, depth-1, False, ai, human, k, r, c)
        undo_move(board, r, c)
        if score > best_score:
            best_score = score
            best_move = (r, c)

    elapsed = time.time() - t0
    algo_name = "Alpha-Beta" if use_alphabeta else "Minimax"
    nodes = alphabeta_nodes if use_alphabeta else minimax_nodes
    print(f"  [{algo_name}] Nút duyệt: {nodes:,} | Thời gian: {elapsed:.4f}s")
    return best_move

# ─────────────────────────────────────────
#  SO SÁNH MINIMAX vs ALPHA-BETA
# ─────────────────────────────────────────
def benchmark(n=5, k=4, depth=3, num_moves=5):
    """
    Thực nghiệm so sánh Minimax thuần vs Alpha-Beta.
    Chơi num_moves nước đầu rồi đo từng thuật toán.
    """
    import copy
    print(f"\n{'='*60}")
    print(f"  BENCHMARK: Minimax vs Alpha-Beta")
    print(f"  Bàn {n}x{n}, thắng {k}, độ sâu {depth}")
    print(f"{'='*60}")

    board = create_board(n)
    # Đi một số nước ngẫu nhiên để tạo trạng thái thực tế
    import random
    players = ['X', 'O']
    idx = 0
    for _ in range(num_moves):
        empty = get_empty_cells(board)
        if not empty:
            break
        r, c = random.choice(empty)
        make_move(board, r, c, players[idx])
        idx = 1 - idx

    print("Trạng thái bàn cờ hiện tại:")
    print_board(board)

    b1 = copy.deepcopy(board)
    b2 = copy.deepcopy(board)

    print("─ Minimax thuần:")
    t0 = time.time()
    global minimax_nodes
    minimax_nodes = 0
    move1 = None
    best1 = -math.inf
    for r, c in get_candidate_moves(b1):
        make_move(b1, r, c, 'X')
        s = minimax(b1, depth-1, False, 'X', 'O', k, r, c)
        undo_move(b1, r, c)
        if s > best1:
            best1 = s
            move1 = (r, c)
    t1 = time.time() - t0
    print(f"  Nút duyệt: {minimax_nodes:,} | Thời gian: {t1:.4f}s | Nước: {move1}")

    print("─ Alpha-Beta Pruning:")
    t0 = time.time()
    global alphabeta_nodes
    alphabeta_nodes = 0
    move2 = None
    best2 = -math.inf
    for r, c in get_candidate_moves(b2):
        make_move(b2, r, c, 'X')
        s = alphabeta(b2, depth-1, False, -math.inf, math.inf, 'X', 'O', k, r, c)
        undo_move(b2, r, c)
        if s > best2:
            best2 = s
            move2 = (r, c)
    t2 = time.time() - t0
    print(f"  Nút duyệt: {alphabeta_nodes:,} | Thời gian: {t2:.4f}s | Nước: {move2}")

    if t2 > 0:
        speedup = t1 / t2
        reduction = (1 - alphabeta_nodes / max(minimax_nodes, 1)) * 100
        print(f"\n  ✓ Alpha-Beta nhanh hơn: {speedup:.1f}x")
        print(f"  ✓ Giảm số nút: {reduction:.1f}%")

# ─────────────────────────────────────────
#  GAME LOOP
# ─────────────────────────────────────────
def play_game(n=5, k=4, depth=4, use_alphabeta=True):
    board = create_board(n)
    algo_name = "Alpha-Beta" if use_alphabeta else "Minimax"
    print(f"\n{'='*50}")
    print(f"  TicTacToe {n}x{n} — Thắng {k} quân")
    print(f"  Thuật toán AI: {algo_name} (độ sâu={depth})")
    print(f"  Người: X | AI: O")
    print(f"{'='*50}\n")
    print_board(board)
    current = 'X'

    while True:
        if current == 'X':
            while True:
                try:
                    inp = input(f"Lượt X — Nhập hàng cột (vd: 2 3): ").split()
                    r, c = int(inp[0]), int(inp[1])
                    if 0 <= r < n and 0 <= c < n and board[r][c] == '.':
                        break
                    print("  Ô không hợp lệ.")
                except:
                    print("  Nhập sai định dạng.")
            make_move(board, r, c, 'X')
        else:
            print(f"AI (O) đang tính [{algo_name}]...")
            r, c = find_best_move(board, depth, 'O', 'X', k, use_alphabeta)
            make_move(board, r, c, 'O')
            print(f"  AI đi: ({r}, {c})")

        print_board(board)

        if check_win(board, r, c, current, k):
            print(f"{'Bạn thắng! 🎉' if current=='X' else 'AI thắng! 🤖'}")
            return
        if is_board_full(board):
            print("Hoà! 🤝")
            return
        current = 'O' if current == 'X' else 'X'

# ─────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────
if __name__ == '__main__':
    print("Bài 4 — Alpha-Beta Pruning TicTacToe NxN")
    print("─" * 40)
    print("1. Người vs AI — Alpha-Beta (5x5)")
    print("2. Người vs AI — Minimax thuần (5x5)")
    print("3. Benchmark so sánh Minimax vs Alpha-Beta (5x5, depth=3)")
    print("4. Benchmark so sánh (10x10, depth=2)")
    choice = input("Chọn (1/2/3/4): ").strip()

    if choice == '1':
        play_game(n=5, k=4, depth=4, use_alphabeta=True)
    elif choice == '2':
        play_game(n=5, k=4, depth=3, use_alphabeta=False)
    elif choice == '3':
        benchmark(n=5, k=4, depth=3, num_moves=4)
    elif choice == '4':
        benchmark(n=10, k=5, depth=2, num_moves=6)
    else:
        benchmark(n=5, k=4, depth=3)

Bài 4 — Alpha-Beta Pruning TicTacToe NxN
────────────────────────────────────────
1. Người vs AI — Alpha-Beta (5x5)
2. Người vs AI — Minimax thuần (5x5)
3. Benchmark so sánh Minimax vs Alpha-Beta (5x5, depth=3)
4. Benchmark so sánh (10x10, depth=2)

  TicTacToe 5x5 — Thắng 4 quân
  Thuật toán AI: Alpha-Beta (độ sâu=4)
  Người: X | AI: O

   0  1  2  3  4
   ───────────────
 0│ .  .  .  .  .
 1│ .  .  .  .  .
 2│ .  .  .  .  .
 3│ .  .  .  .  .
 4│ .  .  .  .  .

   0  1  2  3  4
   ───────────────
 0│ .  X  .  .  .
 1│ .  .  .  .  .
 2│ .  .  .  .  .
 3│ .  .  .  .  .
 4│ .  .  .  .  .

AI (O) đang tính [Alpha-Beta]...
  [Alpha-Beta] Nút duyệt: 11,445 | Thời gian: 0.5675s
  AI đi: (2, 2)
   0  1  2  3  4
   ───────────────
 0│ .  X  .  .  .
 1│ .  .  .  .  .
 2│ .  .  O  .  .
 3│ .  .  .  .  .
 4│ .  .  .  .  .



In [ ]:
"""
Bài 5: Ứng dụng Alpha-Beta Pruning vào Cờ Tướng, Cờ Vua, Cờ Vây
─────────────────────────────────────────────────────────────────
Mỗi trò chơi có:
  - Biểu diễn bàn cờ (board)
  - Hàm sinh nước đi hợp lệ (get_legal_moves)
  - Hàm đánh giá trạng thái (evaluate)
  - AI dùng Alpha-Beta Pruning chung

Chạy: python bai5_board_games.py
"""

import math
import time
import copy
import random

# ═══════════════════════════════════════════════════════════════
#  PHẦN A — ALPHA-BETA PRUNING TỔNG QUÁT (dùng chung)
# ═══════════════════════════════════════════════════════════════
nodes_visited = 0

def alphabeta_general(state, depth, alpha, beta, is_maximizing,
                       get_moves_fn, make_move_fn, undo_move_fn,
                       evaluate_fn, is_terminal_fn):
    """
    Alpha-Beta Pruning tổng quát — không phụ thuộc trò chơi cụ thể.

    Tham số:
      state          : trạng thái bàn cờ (có thể là dict, list,...)
      depth          : độ sâu còn lại
      alpha, beta    : tham số cắt tỉa
      is_maximizing  : True nếu lượt Max (AI)
      get_moves_fn   : fn(state) -> list nước đi
      make_move_fn   : fn(state, move) -> new_state
      undo_move_fn   : không dùng nếu dùng copy (stateless)
      evaluate_fn    : fn(state) -> score (float)
      is_terminal_fn : fn(state) -> (bool, score_or_None)
    """
    global nodes_visited
    nodes_visited += 1

    terminal, term_score = is_terminal_fn(state)
    if terminal:
        return term_score
    if depth == 0:
        return evaluate_fn(state)

    moves = get_moves_fn(state)
    if not moves:
        return evaluate_fn(state)

    if is_maximizing:
        best = -math.inf
        for move in moves:
            new_state = make_move_fn(state, move)
            val = alphabeta_general(new_state, depth-1, alpha, beta, False,
                                    get_moves_fn, make_move_fn, undo_move_fn,
                                    evaluate_fn, is_terminal_fn)
            best = max(best, val)
            alpha = max(alpha, best)
            if beta <= alpha:
                break
        return best
    else:
        best = math.inf
        for move in moves:
            new_state = make_move_fn(state, move)
            val = alphabeta_general(new_state, depth-1, alpha, beta, True,
                                    get_moves_fn, make_move_fn, undo_move_fn,
                                    evaluate_fn, is_terminal_fn)
            best = min(best, val)
            beta = min(beta, best)
            if beta <= alpha:
                break
        return best

def find_best_move_general(state, depth, is_maximizing,
                            get_moves_fn, make_move_fn,
                            evaluate_fn, is_terminal_fn, label=""):
    global nodes_visited
    nodes_visited = 0
    t0 = time.time()
    best_score = -math.inf if is_maximizing else math.inf
    best_move = None

    for move in get_moves_fn(state):
        new_state = make_move_fn(state, move)
        score = alphabeta_general(new_state, depth-1, -math.inf, math.inf,
                                  not is_maximizing,
                                  get_moves_fn, make_move_fn, None,
                                  evaluate_fn, is_terminal_fn)
        if (is_maximizing and score > best_score) or \
           (not is_maximizing and score < best_score):
            best_score = score
            best_move = move

    elapsed = time.time() - t0
    print(f"  [{label}] Nút={nodes_visited:,} | {elapsed:.3f}s | Điểm={best_score:.1f}")
    return best_move


# ═══════════════════════════════════════════════════════════════
#  PHẦN B — CỜ TƯỚNG (đơn giản hóa)
#  Bàn 9x10, một số loại quân cơ bản: Tướng, Xe, Pháo, Mã, Tốt
# ═══════════════════════════════════════════════════════════════

# Ký hiệu: chữ hoa = Đỏ (Max), chữ thường = Đen (Min)
# R=Xe, H=Mã, E=Tượng, A=Sĩ, K=Tướng, C=Pháo, P=Tốt
XIANGQI_INIT = [
    ['r','h','e','a','k','a','e','h','r'],
    ['.','.','.','.','.','.','.','.','.'],
    ['.','c','.','.','.','.','.','c','.'],
    ['p','.','p','.','p','.','p','.','p'],
    ['.','.','.','.','.','.','.','.','.'],
    ['.','.','.','.','.','.','.','.','.'],
    ['P','.','P','.','P','.','P','.','P'],
    ['.','C','.','.','.','.','.','C','.'],
    ['.','.','.','.','.','.','.','.','.'],
    ['R','H','E','A','K','A','E','H','R'],
]

XIANGQI_VALUES = {
    'K': 10000, 'k': -10000,
    'R': 900, 'r': -900,
    'H': 400, 'h': -400,
    'C': 450, 'c': -450,
    'E': 200, 'e': -200,
    'A': 200, 'a': -200,
    'P': 100, 'p': -100,
    '.': 0
}

def xiangqi_copy(state):
    return {
        'board': [row[:] for row in state['board']],
        'turn': state['turn'],
        'red_king': state['red_king'],
        'black_king': state['black_king'],
    }

def xiangqi_init():
    return {
        'board': [row[:] for row in XIANGQI_INIT],
        'turn': 'red',   # red=Max, black=Min
        'red_king': (9, 4),
        'black_king': (0, 4),
    }

def xiangqi_in_bounds(r, c, n_rows=10, n_cols=9):
    return 0 <= r < n_rows and 0 <= c < n_cols

def xiangqi_is_red(p):
    return p.isupper() and p != '.'

def xiangqi_is_black(p):
    return p.islower() and p != '.'

def xiangqi_get_moves(state):
    """Sinh nước đi đơn giản cho Xe (Rook) và Tướng để demo"""
    board = state['board']
    turn = state['turn']
    moves = []

    for r in range(10):
        for c in range(9):
            piece = board[r][c]
            if piece == '.':
                continue
            is_mine = xiangqi_is_red(piece) if turn == 'red' else xiangqi_is_black(piece)
            if not is_mine:
                continue

            p = piece.upper()
            # Xe — đi thẳng
            if p == 'R':
                for dr, dc in [(0,1),(0,-1),(1,0),(-1,0)]:
                    for d in range(1, 10):
                        nr, nc = r + d*dr, c + d*dc
                        if not xiangqi_in_bounds(nr, nc):
                            break
                        target = board[nr][nc]
                        if (turn == 'red' and xiangqi_is_red(target)) or \
                           (turn == 'black' and xiangqi_is_black(target)):
                            break
                        moves.append(((r,c),(nr,nc)))
                        if target != '.':
                            break
            # Tướng — 1 bước trong cung
            elif p == 'K':
                palace = range(7,10) if turn == 'red' else range(0,3)
                for dr, dc in [(0,1),(0,-1),(1,0),(-1,0)]:
                    nr, nc = r+dr, c+dc
                    if nr in palace and 3 <= nc <= 5:
                        target = board[nr][nc]
                        if not ((turn=='red' and xiangqi_is_red(target)) or
                                (turn=='black' and xiangqi_is_black(target))):
                            moves.append(((r,c),(nr,nc)))
            # Tốt
            elif p == 'P':
                if turn == 'red':
                    candidates = [(-1,0)]
                    if r <= 4:
                        candidates += [(0,1),(0,-1)]
                else:
                    candidates = [(1,0)]
                    if r >= 5:
                        candidates += [(0,1),(0,-1)]
                for dr, dc in candidates:
                    nr, nc = r+dr, c+dc
                    if xiangqi_in_bounds(nr, nc):
                        target = board[nr][nc]
                        if not ((turn=='red' and xiangqi_is_red(target)) or
                                (turn=='black' and xiangqi_is_black(target))):
                            moves.append(((r,c),(nr,nc)))
    return moves

def xiangqi_make_move(state, move):
    new = xiangqi_copy(state)
    (r1,c1),(r2,c2) = move
    piece = new['board'][r1][c1]
    new['board'][r2][c2] = piece
    new['board'][r1][c1] = '.'
    if piece == 'K':
        new['red_king'] = (r2, c2)
    elif piece == 'k':
        new['black_king'] = (r2, c2)
    new['turn'] = 'black' if state['turn'] == 'red' else 'red'
    return new

def xiangqi_evaluate(state):
    score = 0
    for row in state['board']:
        for p in row:
            score += XIANGQI_VALUES.get(p, 0)
    return score

def xiangqi_is_terminal(state):
    # Tướng bị bắt
    board = state['board']
    red_alive = any(board[r][c] == 'K' for r in range(10) for c in range(9))
    black_alive = any(board[r][c] == 'k' for r in range(10) for c in range(9))
    if not red_alive:
        return True, -100000
    if not black_alive:
        return True, 100000
    return False, None

def print_xiangqi(state):
    board = state['board']
    print("    " + "  ".join(str(c) for c in range(9)))
    print("   " + "─"*27)
    for r, row in enumerate(board):
        print(f"{r:2d} │ " + "  ".join(p if p != '.' else '·' for p in row))
    print(f"  Lượt: {'Đỏ(Max)' if state['turn']=='red' else 'Đen(Min)'}\n")

def demo_xiangqi(depth=2):
    print(f"\n{'═'*55}")
    print("  CỜ TƯỚNG — Alpha-Beta Pruning Demo")
    print(f"{'═'*55}")
    state = xiangqi_init()
    print_xiangqi(state)

    for move_num in range(1, 7):
        is_max = state['turn'] == 'red'
        side = "Đỏ" if is_max else "Đen"
        print(f"Nước {move_num} — {side}:")
        move = find_best_move_general(
            state, depth, is_max,
            xiangqi_get_moves, xiangqi_make_move,
            xiangqi_evaluate, xiangqi_is_terminal, "Cờ Tướng Alpha-Beta"
        )
        if move is None:
            print("  Không có nước đi hợp lệ!")
            break
        (r1,c1),(r2,c2) = move
        piece = state['board'][r1][c1]
        print(f"  {piece} ({r1},{c1}) → ({r2},{c2})")
        state = xiangqi_make_move(state, move)
        print_xiangqi(state)

        term, score = xiangqi_is_terminal(state)
        if term:
            print(f"Kết thúc! Điểm: {score}")
            break


# ═══════════════════════════════════════════════════════════════
#  PHẦN C — CỜ VUA (đơn giản hóa — chỉ Vua, Hậu, Xe)
# ═══════════════════════════════════════════════════════════════

CHESS_VALUES = {
    'K': 20000, 'k': -20000,
    'Q': 900, 'q': -900,
    'R': 500, 'r': -500,
    'B': 330, 'b': -330,
    'N': 320, 'n': -320,
    'P': 100, 'p': -100,
    '.': 0
}

CHESS_INIT = [
    ['r','n','b','q','k','b','n','r'],
    ['p','p','p','p','p','p','p','p'],
    ['.','.','.','.','.','.','.','.',],
    ['.','.','.','.','.','.','.','.',],
    ['.','.','.','.','.','.','.','.',],
    ['.','.','.','.','.','.','.','.',],
    ['P','P','P','P','P','P','P','P'],
    ['R','N','B','Q','K','B','N','R'],
]

def chess_init():
    return {'board': [row[:] for row in CHESS_INIT], 'turn': 'white'}

def chess_copy(state):
    return {'board': [row[:] for row in state['board']], 'turn': state['turn']}

def chess_is_white(p):
    return p.isupper() and p != '.'
def chess_is_black(p):
    return p.islower() and p != '.'

def chess_get_moves(state):
    board = state['board']
    turn = state['turn']
    moves = []
    n = 8

    def in_b(r, c): return 0 <= r < n and 0 <= c < n

    def slide(r, c, dirs):
        for dr, dc in dirs:
            for d in range(1, n):
                nr, nc = r+d*dr, c+d*dc
                if not in_b(nr, nc): break
                target = board[nr][nc]
                own = chess_is_white if turn=='white' else chess_is_black
                if own(target): break
                moves.append(((r,c),(nr,nc)))
                if target != '.': break

    for r in range(n):
        for c in range(n):
            p = board[r][c]
            if p == '.': continue
            mine = chess_is_white(p) if turn == 'white' else chess_is_black(p)
            if not mine: continue
            pu = p.upper()

            if pu == 'P':
                d = -1 if turn == 'white' else 1
                start = 6 if turn == 'white' else 1
                # Đi thẳng
                if in_b(r+d, c) and board[r+d][c] == '.':
                    moves.append(((r,c),(r+d,c)))
                    if r == start and board[r+2*d][c] == '.':
                        moves.append(((r,c),(r+2*d,c)))
                # Ăn chéo
                for dc in [-1, 1]:
                    nr, nc = r+d, c+dc
                    if in_b(nr, nc):
                        target = board[nr][nc]
                        if (turn=='white' and chess_is_black(target)) or \
                           (turn=='black' and chess_is_white(target)):
                            moves.append(((r,c),(nr,nc)))

            elif pu == 'R':
                slide(r, c, [(0,1),(0,-1),(1,0),(-1,0)])

            elif pu == 'B':
                slide(r, c, [(1,1),(1,-1),(-1,1),(-1,-1)])

            elif pu == 'Q':
                slide(r, c, [(0,1),(0,-1),(1,0),(-1,0),(1,1),(1,-1),(-1,1),(-1,-1)])

            elif pu == 'N':
                for dr, dc in [(-2,-1),(-2,1),(-1,-2),(-1,2),(1,-2),(1,2),(2,-1),(2,1)]:
                    nr, nc = r+dr, c+dc
                    if in_b(nr, nc):
                        target = board[nr][nc]
                        own = chess_is_white if turn=='white' else chess_is_black
                        if not own(target):
                            moves.append(((r,c),(nr,nc)))

            elif pu == 'K':
                for dr in [-1,0,1]:
                    for dc in [-1,0,1]:
                        if dr==0 and dc==0: continue
                        nr, nc = r+dr, c+dc
                        if in_b(nr, nc):
                            target = board[nr][nc]
                            own = chess_is_white if turn=='white' else chess_is_black
                            if not own(target):
                                moves.append(((r,c),(nr,nc)))
    return moves

def chess_make_move(state, move):
    new = chess_copy(state)
    (r1,c1),(r2,c2) = move
    new['board'][r2][c2] = new['board'][r1][c1]
    new['board'][r1][c1] = '.'
    # Phong hậu
    if new['board'][r2][c2] == 'P' and r2 == 0:
        new['board'][r2][c2] = 'Q'
    if new['board'][r2][c2] == 'p' and r2 == 7:
        new['board'][r2][c2] = 'q'
    new['turn'] = 'black' if state['turn'] == 'white' else 'white'
    return new

def chess_evaluate(state):
    score = 0
    for row in state['board']:
        for p in row:
            score += CHESS_VALUES.get(p, 0)
    return score

def chess_is_terminal(state):
    board = state['board']
    white_king = any(board[r][c]=='K' for r in range(8) for c in range(8))
    black_king = any(board[r][c]=='k' for r in range(8) for c in range(8))
    if not white_king: return True, -100000
    if not black_king: return True, 100000
    return False, None

def print_chess(state):
    board = state['board']
    print("    a  b  c  d  e  f  g  h")
    print("   " + "─"*25)
    for r, row in enumerate(board):
        rank = 8 - r
        print(f" {rank} │ " + "  ".join(p if p!='.' else '·' for p in row))
    print(f"  Lượt: {'Trắng(Max)' if state['turn']=='white' else 'Đen(Min)'}\n")

def demo_chess(depth=2):
    print(f"\n{'═'*55}")
    print("  CỜ VUA — Alpha-Beta Pruning Demo")
    print(f"{'═'*55}")
    state = chess_init()
    print_chess(state)

    for move_num in range(1, 7):
        is_max = state['turn'] == 'white'
        side = "Trắng" if is_max else "Đen"
        print(f"Nước {move_num} — {side}:")
        move = find_best_move_general(
            state, depth, is_max,
            chess_get_moves, chess_make_move,
            chess_evaluate, chess_is_terminal, "Cờ Vua Alpha-Beta"
        )
        if move is None:
            print("  Hết nước đi!")
            break
        (r1,c1),(r2,c2) = move
        cols = 'abcdefgh'
        piece = state['board'][r1][c1]
        print(f"  {piece}: {cols[c1]}{8-r1} → {cols[c2]}{8-r2}")
        state = chess_make_move(state, move)
        print_chess(state)

        term, score = chess_is_terminal(state)
        if term:
            print(f"Kết thúc! {'Trắng thắng' if score>0 else 'Đen thắng'}")
            break


# ═══════════════════════════════════════════════════════════════
#  PHẦN D — CỜ VÂY (Go, 9x9) với MCTS đơn giản
#  Alpha-Beta không phù hợp cho cờ vây (nhánh cực lớn),
#  ta demo Alpha-Beta + heuristic đếm vùng kiểm soát,
#  và giải thích tại sao MCTS được dùng thực tế hơn.
# ═══════════════════════════════════════════════════════════════

def go_init(n=9):
    return {
        'board': [['.' for _ in range(n)] for _ in range(n)],
        'turn': 'B',   # B=Đen(Max), W=Trắng(Min)
        'n': n,
        'last_move': None,
        'pass_count': 0,
    }

def go_copy(state):
    return {
        'board': [row[:] for row in state['board']],
        'turn': state['turn'],
        'n': state['n'],
        'last_move': state['last_move'],
        'pass_count': state['pass_count'],
    }

def go_get_moves(state):
    """Trả về tất cả ô trống + nước pass"""
    n = state['n']
    board = state['board']
    moves = []
    for r in range(n):
        for c in range(n):
            if board[r][c] == '.':
                moves.append((r, c))
    moves.append('pass')
    # Giới hạn số nước để demo không quá chậm
    return moves[:20] + ['pass']

def go_make_move(state, move):
    new = go_copy(state)
    if move == 'pass':
        new['pass_count'] = state['pass_count'] + 1
        new['turn'] = 'W' if state['turn'] == 'B' else 'B'
        return new
    r, c = move
    new['board'][r][c] = state['turn']
    new['pass_count'] = 0
    new['last_move'] = move
    new['turn'] = 'W' if state['turn'] == 'B' else 'B'
    return new

def go_evaluate(state):
    """
    Heuristic đơn giản: đếm số quân + ảnh hưởng vùng (territory)
    Thực tế dùng Neural Network như AlphaGo/KataGo
    """
    board = state['board']
    n = state['n']
    score = 0
    black_count = sum(board[r][c] == 'B' for r in range(n) for c in range(n))
    white_count = sum(board[r][c] == 'W' for r in range(n) for c in range(n))
    # Điểm kiểm soát: ô trống gần quân nhiều hơn
    for r in range(n):
        for c in range(n):
            if board[r][c] == '.':
                b_near = w_near = 0
                for dr, dc in [(0,1),(0,-1),(1,0),(-1,0)]:
                    nr, nc = r+dr, c+dc
                    if 0 <= nr < n and 0 <= nc < n:
                        if board[nr][nc] == 'B': b_near += 1
                        elif board[nr][nc] == 'W': w_near += 1
                if b_near > w_near: score += 0.5
                elif w_near > b_near: score -= 0.5
    score += black_count - white_count
    return score

def go_is_terminal(state):
    if state['pass_count'] >= 2:
        score = go_evaluate(state)
        if score > 0: return True, 10000
        elif score < 0: return True, -10000
        return True, 0
    return False, None

def print_go(state):
    board = state['board']
    n = state['n']
    print("    " + "  ".join(str(c) for c in range(n)))
    print("   " + "─"*(n*3))
    for r in range(n):
        row_str = "  ".join({'B':'●','W':'○','.':'·'}[board[r][c]] for c in range(n))
        print(f"{r:2d} │ {row_str}")
    b = sum(board[r][c]=='B' for r in range(n) for c in range(n))
    w = sum(board[r][c]=='W' for r in range(n) for c in range(n))
    print(f"  ● Đen={b}  ○ Trắng={w}  Lượt: {'Đen(Max)' if state['turn']=='B' else 'Trắng(Min)'}\n")

def demo_go(depth=2, moves=8):
    print(f"\n{'═'*55}")
    print("  CỜ VÂY 9x9 — Alpha-Beta Pruning Demo")
    print("  (Thực tế: MCTS + Neural Net như AlphaGo)")
    print(f"{'═'*55}")
    state = go_init(9)
    print_go(state)

    for move_num in range(1, moves+1):
        is_max = state['turn'] == 'B'
        side = "Đen ●" if is_max else "Trắng ○"
        print(f"Nước {move_num} — {side}:")
        move = find_best_move_general(
            state, depth, is_max,
            go_get_moves, go_make_move,
            go_evaluate, go_is_terminal, "Cờ Vây Alpha-Beta"
        )
        if move is None:
            print("  Không có nước đi!")
            break
        if move == 'pass':
            print("  PASS")
        else:
            print(f"  Đi: ({move[0]}, {move[1]})")
        state = go_make_move(state, move)
        print_go(state)

        term, score = go_is_terminal(state)
        if term:
            print(f"Kết thúc! {'Đen thắng' if score>0 else 'Trắng thắng' if score<0 else 'Hoà'}")
            break


# ═══════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════
if __name__ == '__main__':
    print("Bài 5 — Alpha-Beta cho Cờ Tướng, Cờ Vua, Cờ Vây")
    print("─" * 50)
    print("1. Demo Cờ Tướng (depth=2)")
    print("2. Demo Cờ Vua (depth=2)")
    print("3. Demo Cờ Vây 9x9 (depth=2)")
    print("4. Demo tất cả")
    choice = input("Chọn (1/2/3/4): ").strip()

    if choice == '1' or choice == '4':
        demo_xiangqi(depth=2)
    if choice == '2' or choice == '4':
        demo_chess(depth=2)
    if choice == '3' or choice == '4':
        demo_go(depth=2, moves=8)
    if choice not in ['1','2','3','4']:
        demo_chess(depth=2)

Bài 5 — Alpha-Beta cho Cờ Tướng, Cờ Vua, Cờ Vây
──────────────────────────────────────────────────
1. Demo Cờ Tướng (depth=2)
2. Demo Cờ Vua (depth=2)
3. Demo Cờ Vây 9x9 (depth=2)
4. Demo tất cả
Chọn (1/2/3/4): 1

═══════════════════════════════════════════════════════
  CỜ TƯỚNG — Alpha-Beta Pruning Demo
═══════════════════════════════════════════════════════
    0  1  2  3  4  5  6  7  8
   ───────────────────────────
 0 │ r  h  e  a  k  a  e  h  r
 1 │ ·  ·  ·  ·  ·  ·  ·  ·  ·
 2 │ ·  c  ·  ·  ·  ·  ·  c  ·
 3 │ p  ·  p  ·  p  ·  p  ·  p
 4 │ ·  ·  ·  ·  ·  ·  ·  ·  ·
 5 │ ·  ·  ·  ·  ·  ·  ·  ·  ·
 6 │ P  ·  P  ·  P  ·  P  ·  P
 7 │ ·  C  ·  ·  ·  ·  ·  C  ·
 8 │ ·  ·  ·  ·  ·  ·  ·  ·  ·
 9 │ R  H  E  A  K  A  E  H  R
  Lượt: Đỏ(Max)

Nước 1 — Đỏ:
  [Cờ Tướng Alpha-Beta] Nút=110 | 0.004s | Điểm=0.0
  P (6,0) → (5,0)
    0  1  2  3  4  5  6  7  8
   ───────────────────────────
 0 │ r  h  e  a  k  a  e  h  r
 1 │ ·  ·  ·  ·  ·  ·  ·  ·  ·
 2 │ ·  c  ·  ·  ·  ·  ·  c  ·
 3 │ p  ·  p